# Test notebook for the BHoM user guide: from a flat table to gold, with Genie Code

This notebook tests, **on Databricks**, the whole flow of [User guide 1](https://github.com/BHoM/BHoM_PythonSchema/blob/develop/docs/User-Guide-1-From-Flat-Tables-To-Gold.md) on a small sample table:

| Section | What you do | What is tested |
|---|---|---|
| 1 | Install the package | `bhom-schema[mapping]` installs and imports in your workspace |
| 2, 3 | Your variables, then make a flat silver table | You can create tables in your schemas |
| 4 | **Genie Code:** check alignment | The `@bhom-mapping` skill is published and finds the table |
| 5 | **Genie Code:** create the mapping file, with `Project` as the root | Genie Code can map the columns and the file is valid |
| 6 | **Genie Code:** write gold | Gold tables are written, linked by `Project_id` and `Building_id` |
| 7 | Read gold back into a flat DataFrame | The joins along the links rebuild the flat table |
| 8 | A column that has no BHoM home | The schema search, and where to go next ([guide 2](https://github.com/BHoM/BHoM_PythonSchema/blob/develop/docs/User-Guide-2-Extending-The-Schema.md)) |
| 9 | Clean up | The demo tables and the mapping file are dropped (always) |

The cells that follow each Genie Code step **check what Genie Code did** and say exactly what is missing, so a run that ends without an error means the flow works. The notebook tests the Genie Code way only: the mapping and the gold tables are made by Genie Code, never by the notebook.

## How to upload and run this notebook

**Upload** (pick one):

- **Import the file.** In Databricks open **Workspace**, go to your own folder (for example `/Workspace/Users/<you>/bhom-test`), then **⋮** (or *Create*) → **Import** and choose this `.ipynb` file.
- **Git folder.** In **Workspace**, *Create* → **Git folder**, paste `https://github.com/BHoM/BHoM_PythonSchema`, and open `.ci/unit-tests/notebooks/Databricks-User-Guide-Test.ipynb`.

**Run**

1. Attach **serverless compute**, or a cluster with Python 3.12 or later. You need permission to create tables in two schemas of one catalog (the schemas must already exist).
2. In section 2, change the three variables `CATALOG`, `SILVER_SCHEMA` (silver tables) and `GOLD_SCHEMA` (gold tables) to your own, then run the cell. The `@bhom-mapping` skill must be published to your workspace (ask a workspace admin if it is not).
3. Run the cells **in order, one section at a time**, not *Run all*: the notebook pauses at the three Genie Code steps, and you must do each one in the Genie Code panel before running the check cell under it. Read the **Expect** lines.
4. The mapping file is kept in a `bhom-mappings` folder next to this notebook. In a Git folder that makes a file you can delete or ignore; do not commit it.
5. Run section 9 at the end: it **drops** the demo tables and the mapping file again.

Nothing in the notebook is expected to fail. The silver table is named `bhom_user_guide_test_flat`, and the gold tables are named after the entities of the mapping (`Project`, `Building`, `Storey`, `Beam`). Because section 9 drops them, **use a gold schema that holds nothing else**.

## 1. Install the package

`%pip` and the restart each go in a cell of their own. Packages installed this way last for this notebook session, so this is also how you update the package later.

In [ ]:
%pip install --upgrade --pre --no-cache-dir --index-url https://test.pypi.org/simple/ --extra-index-url https://pypi.org/simple/ "bhom-schema[mapping]"

In [ ]:
dbutils.library.restartPython()

**Expect:** a package version such as `10.0.0a2611201430` (a development build), the BHoM version `10.0`, and the first line of the `Project` class card. If the import fails, the install cell did not run or Python was not restarted.

In [ ]:
import bhom_schema
from bhom_schema.mapping import describe_class

print("package version :", bhom_schema.__version__)
print("BHoM version    :", bhom_schema.__bhom_version__)
print(describe_class("Project").text().splitlines()[0])

## 2. Settings

Change the three variables at the top of the cell, then run it. **Use a catalog and schemas of your own**: both schemas must already exist and you must be allowed to create tables in them (the values are only an example). Section 9 drops the tables this notebook makes, so the gold schema must hold nothing else.

In [ ]:
import os

# Change these three to a catalog and schemas of your own
CATALOG = "student_area"
SILVER_SCHEMA = "vcl"
GOLD_SCHEMA = "vcl_gold"

SILVER = f"{CATALOG}.{SILVER_SCHEMA}.bhom_user_guide_test_flat"
GOLD = f"{CATALOG}.{GOLD_SCHEMA}"
MAPPING_FOLDER = os.getcwd()

# Fails here, with a clear message, when a schema does not exist or cannot be read
for schema_name in (SILVER_SCHEMA, GOLD_SCHEMA):
    spark.sql(f"DESCRIBE SCHEMA `{CATALOG}`.`{schema_name}`")

print("silver table  :", SILVER)
print("gold schema   :", GOLD)
print("mapping folder:", MAPPING_FOLDER)

## 3. A flat silver table

In real work you read the Parquet files of the bronze volume and flatten them (see step 1 of the guide). Here a small sample stands in. It is like real data: the project and model repeat on every row, storey heights are in **millimetres**, materials are **free text**, a `source_file` column keeps the lineage, and one column, `number_of_lifts`, will turn out to have **no BHoM property** (section 8).

There are two projects with two models each and three beam families per model. **Expect:** twelve rows written to the silver table, and any mapping file of an earlier run of this notebook removed, so that this run starts clean.

In [ ]:
from bhom_schema.mapping import find_mapping
from bhom_schema.tables.sql import quote_table

SILVER_DDL = (
    "`project_number` STRING, `project_name` STRING, `model_name` STRING, `levels_above_ground` INT, `typical_level_gap` DOUBLE, "
    "`structural_material` STRING, `family_type` STRING, `number_of_lifts` INT, `source_file` STRING")

def sample_rows():
    """Returns the rows of the sample silver table: two projects, two models each, three beam families per model."""
    rows = []
    for number, name in (("12345", "Alpha Tower"), ("67890", "Beta Hall")):
        for model, levels, height, lifts in (("Tower", 24, 3900.0, 6), ("Podium", 4, 4200.0, 2)):
            for index, family in enumerate(("UB-0", "UB-1", "UB-2")):
                rows.append({
                    "project_number": number, "project_name": name, "model_name": model, "levels_above_ground": levels, "typical_level_gap": height,
                    "structural_material": "glulam" if index == 1 else "Steel - hot rolled", "family_type": family, "number_of_lifts": lifts,
                    "source_file": f"{number}/{model}/{family}.parquet"})

    return rows

# A mapping file left by an earlier run belongs to this demo table only: remove it, so that this run starts clean
stale = find_mapping(SILVER, start=MAPPING_FOLDER)
if stale is not None:
    stale.unlink()
    print("removed the mapping file of an earlier run:", stale)

# Section 9 drops the gold tables of this notebook, so they must not be somebody else's
GOLD_NAMES = ("Project", "Building", "Storey", "Beam")
existing = [f"{GOLD}.{name}" for name in GOLD_NAMES if spark.catalog.tableExists(f"{GOLD}.{name}")]
if existing:
    raise AssertionError(f"these tables exist already: {', '.join(existing)}. Use a gold schema that holds nothing else, or drop them if an earlier run of this notebook left them")

SILVER_ROWS = sample_rows()
spark.createDataFrame(SILVER_ROWS, schema=SILVER_DDL).write.mode("overwrite").option("overwriteSchema", "true").saveAsTable(quote_table(SILVER))
print(f"wrote {len(SILVER_ROWS)} rows to {SILVER}")

## 4. Check alignment

### Genie Code step 1

Open the **Genie Code panel** next to the notebook, start a **new chat**, and type (with your table name; it is printed by section 2):

```
@bhom-mapping check <catalog>.<schema>.bhom_user_guide_test_flat
```

**Expect** an answer in a few lines: the table **has no mapping yet**, so it is not BHoM-aligned, with an offer to create the mapping. If `@bhom-mapping` is not listed, the skill is not published to this workspace: ask a workspace admin. **Do not accept the offer yet:** the next cell shows the same answer from code, and section 5 is where you map.

The cell below asks the same question in code, so you can compare. **Expect:** `... has no mapping yet, so it is not BHoM-aligned.`

In [ ]:
from bhom_schema.mapping import alignment

print(alignment(spark, SILVER, start=MAPPING_FOLDER).summary())

## 5. Create the mapping file

### Genie Code step 2

In the Genie Code panel, ask:

```
@bhom-mapping map <catalog>.<schema>.bhom_user_guide_test_flat. Use Project as the root entity:
key it on project_number, make Building a child of Project, and make every other entity a child of Building.
```

Genie Code should ask which folder is your tool's folder: answer with the folder of **this notebook** (it is printed as `mapping folder` by section 2). Then it should work like this, and **you** decide at each point:

- It profiles the table and the BHoM classes, and **proposes the entities** with their keys: `Project` (key `project_number`), `Building` (`project_number`, `model_name`), `Storey`, and `Beam` (`project_number`, `model_name`, `family_type`).
- It **asks about the unit** of `typical_level_gap` (the values are in millimetres), and about any text value it is unsure of.
- It **shows the mapping file as a diff and waits**: nothing is saved before you approve. Read it. Check the keys, the `unit: mm`, the material text map (`glulam` is `TimberGlulam`, `Steel - hot rolled` is `SteelHotRolled`), and that `source_file` and `number_of_lifts` are under `unmapped` with a reason.
- It runs `validate_mapping` and explains any problem. Approve its fix or correct it.

If the answer differs from what you expect, tell Genie Code what is wrong, as you would in real work. **Once agreed, tell Genie to save the mapping file. Then run the check cell below.**

### Check the mapping

This cell checks what Genie Code saved in the mapping file. It fails with a list of what is missing:

- the file is found from this notebook's folder, and **loads** (every rule of the file format);
- there is a `Project` entity, a `Building` entity, and **every other entity is linked to the Project** through its `parent` chain;
- every column of the table is **accounted for** (used or listed under `unmapped`), and the rows **validate**.

**Expect:** the entities with their keys and parents, where each of `source_file` and `number_of_lifts` went, the line `the mapping is aligned and valid`, and the report of `validate_mapping`.

In [ ]:
from bhom_schema.mapping import MappingError, load_mapping, validate_mapping

def check_mapping():
    """Checks the mapping file of the silver table against the rules of the guide, and returns it. Raises AssertionError listing everything that is wrong."""
    problems = []
    path = find_mapping(SILVER, start=MAPPING_FOLDER)
    if path is None:
        raise AssertionError(f"no mapping file was found for {SILVER} from {MAPPING_FOLDER}. Genie Code may have saved it in another folder: ask it where, and move it to {MAPPING_FOLDER}/bhom-mappings/")

    try:
        mapping = load_mapping(SILVER, start=MAPPING_FOLDER)
    except MappingError as error:
        raise AssertionError(f"{path} breaks the rules of the mapping file:\n{error}") from error

    by_name = {entity.name: entity for entity in mapping.entities}
    classes = {entity.name: entity.class_path.rsplit(".", 1)[1] for entity in mapping.entities}
    for name, entity in by_name.items():
        print(f"  {name:10} {classes[name]:10} key {', '.join(entity.key):45} parent {entity.parent or '-'}")

    roots = [name for name, kind in classes.items() if kind == "Project"]
    if not roots:
        problems.append("there is no entity of the class Project: ask Genie Code to use Project as the root entity, keyed on project_number")

    if "Building" not in classes.values():
        problems.append("there is no entity of the class Building")

    for name, entity in by_name.items():
        chain, current = [], entity
        while current.parent is not None and current.parent in by_name and current.parent not in chain:
            chain.append(current.parent)
            current = by_name[current.parent]

        if classes[current.name] != "Project":
            problems.append(f"{name} is not linked to a Project: its chain of parents ends at {current.name}")

    for column in ("source_file", "number_of_lifts"):
        if column in mapping.unmapped:
            print(f"  {column}: unmapped, because: {mapping.unmapped[column]}")
        elif column in mapping.used_columns():
            print(f"  {column}: mapped to a property")
        else:
            problems.append(f"{column} is neither mapped nor listed under unmapped")

    found = alignment(spark, SILVER, start=MAPPING_FOLDER)
    if not found.aligned:
        problems.append(found.summary())

    report = validate_mapping(spark, SILVER, mapping)
    print()
    print(report.summary())
    if not report.ok:
        problems.append("the mapping is not valid for the rows (see the report above)")

    if problems:
        raise AssertionError("the mapping is not what the guide describes:\n- " + "\n- ".join(problems))

    print()
    print("the mapping is aligned and valid:", path)
    return mapping

MAPPING = check_mapping()

## 6. Write the gold tables

### Genie Code step 3

In the Genie Code panel, ask:

```
@bhom-mapping write gold <catalog>.<schema>.bhom_user_guide_test_flat
```

**Expect:** it validates all rows, writes one table per entity in your gold schema, and reports the tables with their row counts: 2 projects, 4 buildings, 4 storeys and 12 beams (the entity names may differ). It writes only because you asked.

### Check the gold tables

This cell checks every gold table of the mapping: it exists, it has **one row per object** of the silver table (the groups of its key), each row has its **link to the parent** (`Project_id`, `Building_id`) and the link points at a row that exists. **Expect:** one line for each table that ends in `ok`.

In [ ]:
from bhom_schema.mapping import check_key
from bhom_schema.tables.spark import read_rows

def check_gold(mapping):
    """Checks the gold table of every entity against the silver table, and returns the gold rows by entity. Raises AssertionError listing everything that is wrong."""
    problems = []
    rows_of = {}
    for entity in mapping.entities:
        table = f"{mapping.gold_schema}.{entity.name}"
        if not spark.catalog.tableExists(table):
            problems.append(f"{table} does not exist: has the gold step been done?")
            continue

        rows_of[entity.name] = read_rows(spark, table)

    for entity in mapping.entities:
        if entity.name not in rows_of:
            continue

        rows = rows_of[entity.name]
        expected = check_key(spark, SILVER, list(entity.key)).groups
        notes = []
        if len(rows) != expected:
            problems.append(f"{entity.name}: {len(rows)} rows in gold, but its key {', '.join(entity.key)} makes {expected} objects in silver")

        if entity.parent is not None and entity.parent in rows_of:
            link = f"{entity.parent}_id"
            parent_ids = {row["BHoM_Guid"] for row in rows_of[entity.parent]}
            unlinked = [row for row in rows if row.get(link) not in parent_ids]
            if unlinked:
                problems.append(f"{entity.name}: {len(unlinked)} rows have no {link} that matches a row of {entity.parent}")

            notes.append(f"{link} -> {entity.parent}")

        print(f"  {entity.name:10} {len(rows):3} rows  {', '.join(notes):28} {'ok' if len(rows) == expected else 'NOT OK'}")

    if problems:
        raise AssertionError("the gold tables are not what the guide describes:\n- " + "\n- ".join(problems))

    return rows_of

GOLD_ROWS = check_gold(MAPPING)

## 7. Read gold back into a flat DataFrame

Every object points to its parent, and the chain ends at the Project, so the gold tables can be joined back into one wide table. `gold_table` reads a gold table with every column prefixed by its entity (so `Name` does not clash), its `BHoM_Guid` as `<Entity>_id`, and the link to its parent unchanged. The flat table has **one row per beam**, with the building, the project and the storey of that beam.

**Expect:** a table with the project number and name, the building, the number of storeys, the storey height **in metres** (3.9 and 4.2: the silver values were millimetres), the beam size and its material (enum member names), and the checks under it. The cells of this section need Spark joins, which is why the unit tests do not run them.

In [ ]:
# The entity of each class in this mapping: Genie Code may have named them differently from the guide
NAME_OF = {entity.class_path.rsplit(".", 1)[1]: entity.name for entity in MAPPING.entities}
beam, building, project, storey = (NAME_OF[name] for name in ("Beam", "Building", "Project", "Storey"))

In [ ]:
# databricks-only: uses Spark joins
from pyspark.sql import functions as F

def gold_table(entity):
    """Reads a gold table with every column prefixed by the entity (Beam_TypicalSize), its BHoM_Guid as <Entity>_id, and the link to its parent unchanged (Building_id)."""
    frame = spark.table(quote_table(f"{GOLD}.{entity}")).drop("_bhomVersion", "Tags")
    columns = []
    for name in frame.columns:
        if name == "BHoM_Guid":
            columns.append(F.col(name).alias(f"{entity}_id"))
        elif name.endswith("_id"):
            columns.append(F.col(name))
        else:
            columns.append(F.col(name).alias(f"{entity}_{name}"))

    return frame.select(*columns)

flat = (
    gold_table(beam)
    .join(gold_table(building), f"{building}_id", "left")
    .join(gold_table(project), f"{project}_id", "left")
    .join(gold_table(storey), f"{building}_id", "left"))

display(flat.select(
    f"{project}_Number", f"{project}_Name", f"{building}_Name", f"{building}_StoreysAboveGround", f"{storey}_TypicalHeight", f"{beam}_TypicalSize", f"{beam}_TypicalMaterial"))

In [ ]:
# databricks-only: uses Spark joins
problems = []
beams = len(GOLD_ROWS[beam])
if flat.count() != beams:
    problems.append(f"the flat table has {flat.count()} rows, but there are {beams} beams: a join multiplied or lost rows")

if flat.filter(F.col(f"{project}_Number").isNull()).count():
    problems.append("some beams have no project number: a link from a beam to its building or from a building to its project is broken")

heights = sorted(row[0] for row in flat.select(f"{storey}_TypicalHeight").distinct().collect())
if heights != [3.9, 4.2]:
    problems.append(f"the storey heights are {heights}, expected [3.9, 4.2] metres (silver holds them in millimetres)")

if problems:
    raise AssertionError("the flat table is not what the guide describes:\n- " + "\n- ".join(problems))

print(f"the flat table has {beams} rows, one per beam, every beam has its project, and the heights are {heights} m")

The same gold rows can be read back as BHoM **objects** instead (enums become Python enum members, lengths are metres). **Expect:** two buildings' storey heights and the first beams with their materials.

In [ ]:
from bhom_schema.DataSchema_oM.Elements.Beam import Beam
from bhom_schema.DataSchema_oM.Project import Project
from bhom_schema.tables.spark import read_objects

projects = read_objects(spark, Project, f"{GOLD}.{project}", fields=MAPPING.entity(project).fields, keyed=True)
for item in projects.values():
    print(f"project {item.Number}: {item.Name}")

for item in read_objects(spark, Beam, f"{GOLD}.{beam}", fields=MAPPING.entity(beam).fields, limit=4):
    print(f"beam {item.TypicalSize:5} {item.TypicalMaterial.name if item.TypicalMaterial else None}")

## 8. A column that is not mapped

`number_of_lifts` has no property in the BHoM schema (at the time of writing), so it is listed under `unmapped` and stays in silver. The guide gives three cases for such a column: lineage that stays in silver, a column Genie was unsure about, or a column that **belongs in the schema**. The number of lifts is the third. This cell searches the schema for it.

**Expect:** no property found for "lift" or "elevator" (and the reason for the unmapped column). If a property is found, someone has added it since: the schema has it, so `number_of_lifts` can be mapped, which is the end of [guide 2](https://github.com/BHoM/BHoM_PythonSchema/blob/develop/docs/User-Guide-2-Extending-The-Schema.md).

In [ ]:
from bhom_schema.mapping import search_properties

for words in ("lift", "elevator"):
    found = search_properties(words)
    print(f"properties that mention {words!r}:", ", ".join(match.label for match in found) or "none")

print()
print("number_of_lifts in the mapping:", MAPPING.unmapped.get("number_of_lifts", "mapped to a property"))

**Optional: test guide 2 with this column.** Follow [guide 2](https://github.com/BHoM/BHoM_PythonSchema/blob/develop/docs/User-Guide-2-Extending-The-Schema.md) to propose a `NumberOfLifts` property on `Building`, and come back when its build has been published. Then run section 1 again (it updates the package), re-run section 8 to see the property, and in the Genie Code panel ask `@bhom-mapping check ...` and then `@bhom-mapping map ...` for this table: it should map `number_of_lifts` to the new property. **Do not merge a pull request only to test this.**

## 9. Clean up

Run this cell to drop the silver table, the gold tables and the mapping file of this notebook. Look at the tables in the catalog explorer first if you want to: they are gone after this cell.

In [ ]:
names = [SILVER] + [f"{MAPPING.gold_schema}.{entity.name}" for entity in MAPPING.entities]
path = find_mapping(SILVER, start=MAPPING_FOLDER)

for name in names:
    spark.sql(f"DROP TABLE IF EXISTS {quote_table(name)}")
    print("dropped", name)

if path is not None and path.exists():
    path.unlink()
    print("removed", path)

## Result

If every check cell passed, then in your workspace: the package installs, the `bhom-mapping` skill is published and works, a mapping with a `Project` root can be written and validated, gold tables are written and linked, and they read back into a flat DataFrame with the project on every row. Anything you had to correct in Genie Code's answer is useful feedback for the skill: tell Viet Le, with the prompt you used.

The guides: [User guide 1](https://github.com/BHoM/BHoM_PythonSchema/blob/develop/docs/User-Guide-1-From-Flat-Tables-To-Gold.md) (the workflow) and [User guide 2](https://github.com/BHoM/BHoM_PythonSchema/blob/develop/docs/User-Guide-2-Extending-The-Schema.md) (adding what is missing to the schema).